# Semana 9: Aprendizaje No Supervisado: Clustering y PCA
## Notebook de Ejercicios (NB2) – Segmentación de Clientes

**Propósito:** Aplicar técnicas de clustering y reducción de dimensionalidad a un problema real de segmentación de clientes.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Aplicar K-Means para segmentar clientes y determinar el número óptimo de clusters.
- Visualizar los segmentos en 2D usando PCA.
- Interpretar los perfiles de cada cluster analizando las medias de las variables.
- Probar DBSCAN y comparar resultados, identificando posibles outliers.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: Mall Customers

El dataset **Mall Customers** contiene información de clientes de un centro comercial:
- `CustomerID`: Identificador único
- `Gender`: Género
- `Age`: Edad
- `Annual Income (k$)`: Ingreso anual en miles de dólares
- `Spending Score (1-100)`: Puntaje de gasto asignado por el centro comercial

Cargamos desde una URL pública.

In [ ]:
# URL del dataset (Kaggle Mall Customers)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/Mall_Customers.csv'

try:
    df = pd.read_csv(url)
    print("Dataset cargado correctamente.")
except:
    print("No se pudo cargar desde URL. Usando datos locales de muestra...")
    # Datos de muestra mínimos para que el notebook no falle
    from sklearn.datasets import make_blobs
    X_sample, _ = make_blobs(n_samples=200, centers=4, random_state=42)
    df = pd.DataFrame(X_sample, columns=['Annual Income', 'Spending Score'])
    df['Age'] = np.random.randint(18, 70, 200)
    df['Gender'] = np.random.choice(['Male', 'Female'], 200)

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información general
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

---
## 2. Preprocesamiento

Seleccionamos las variables numéricas para el clustering: `Age`, `Annual Income (k$)`, `Spending Score (1-100)`.

Estandarizamos los datos para que todas las variables tengan la misma influencia.

In [ ]:
# Seleccionamos variables para clustering
if 'Annual Income (k$)' in df.columns:
    feature_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
else:
    # Si es el dataset sintético, usamos las columnas disponibles
    feature_cols = ['Age', 'Annual Income', 'Spending Score']
    df.columns = ['CustomerID', 'Gender', 'Age', 'Annual Income', 'Spending Score'] if 'CustomerID' in df.columns else df.columns

X = df[feature_cols].copy()

# Estandarizamos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables seleccionadas:", feature_cols)
print("\nPrimeras 5 filas escaladas:")
pd.DataFrame(X_scaled, columns=feature_cols).head()

---
## 3. PCA para Visualización

Reducimos la dimensionalidad a 2 componentes para visualizar los clusters.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:", pca.explained_variance_ratio_)
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_):.2%}")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Datos proyectados en 2D con PCA')
plt.grid(True)
plt.show()

---
## 4. K-Means: Determinación del Número Óptimo de Clusters

Usamos el método del codo y el coeficiente de silueta para elegir k.

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].grid(True)

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Coeficiente de Silueta')
axes[1].set_title('Silueta vs k')
axes[1].grid(True)

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Mejor k según silueta: {best_k}")
print(f"Silueta promedio para k={best_k}: {silhouettes[best_k-2]:.4f}")

## 5. Aplicación de K-Means con k Óptimo

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_kmeans = kmeans_final.fit_predict(X_scaled)

df['Cluster_KMeans'] = labels_kmeans

print(f"Distribución de clusters:")
print(df['Cluster_KMeans'].value_counts().sort_index())

### 5.1. Visualización de los clusters en el espacio PCA

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title(f'Clusters de K-Means (k={best_k}) visualizados con PCA')
plt.grid(True)
plt.show()

## 6. Interpretación de los Perfiles de los Clusters

Calculamos la media de cada variable por cluster para entender sus características.

In [ ]:
# Perfil de clusters (en escala original)
cluster_profile = df.groupby('Cluster_KMeans')[feature_cols].mean()
print("=== Perfil de Clusters (medias en escala original) ===")
cluster_profile.round(1)

In [ ]:
# Visualización de perfiles
fig, axes = plt.subplots(1, len(feature_cols), figsize=(15, 4))

for i, col in enumerate(feature_cols):
    cluster_profile[col].plot(kind='bar', ax=axes[i], color=plt.cm.viridis(np.linspace(0, 1, best_k)))
    axes[i].set_title(f'Media de {col} por cluster')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel(col)
    axes[i].grid(axis='y')

plt.tight_layout()
plt.show()

### 6.1. Descripción de los Clusters

Basado en las medias, podemos caracterizar cada cluster. Por ejemplo:
- **Cluster 0**: Clientes jóvenes con ingresos medios-altos y alto puntaje de gasto.
- **Cluster 1**: Clientes mayores con ingresos altos pero bajo gasto.
- **Cluster 2**: Clientes jóvenes con bajos ingresos y bajo gasto.
- ...

Esta interpretación es clave para definir estrategias de marketing personalizadas.

---
## 7. DBSCAN: Clustering Basado en Densidad

Aplicamos DBSCAN para comparar resultados y detectar posibles outliers.

In [ ]:
# Probamos diferentes valores de eps
eps_values = [0.3, 0.5, 0.7, 1.0]
min_samples = 5

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels_dbscan = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    n_noise = list(labels_dbscan).count(-1)
    
    axes[i].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_dbscan, cmap='viridis', alpha=0.6)
    axes[i].set_title(f'DBSCAN: eps={eps}, min_samples={min_samples}\nClusters={n_clusters}, Ruido={n_noise}')
    axes[i].set_xlabel('PC1')
    axes[i].set_ylabel('PC2')

plt.tight_layout()
plt.show()

### 7.1. Selección del mejor parámetro para DBSCAN

Elegimos el valor de eps que produce una segmentación razonable (por ejemplo, eps=0.5) y comparamos con K-Means.

In [ ]:
dbscan_final = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan_final = dbscan_final.fit_predict(X_scaled)

df['Cluster_DBSCAN'] = labels_dbscan_final

print("Distribución de clusters DBSCAN:")
print(df['Cluster_DBSCAN'].value_counts().sort_index())

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.6)
axes[0].set_title(f'K-Means (k={best_k})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_dbscan_final, cmap='viridis', alpha=0.6)
axes[1].set_title('DBSCAN (eps=0.5, min_samples=5)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

### 7.2. Análisis de Outliers detectados por DBSCAN

DBSCAN marca como ruido (label = -1) los puntos que no pertenecen a ningún cluster denso. Estos pueden ser outliers interesantes.

In [ ]:
outliers = df[df['Cluster_DBSCAN'] == -1]
print(f"Número de outliers detectados: {len(outliers)}")

if len(outliers) > 0:
    print("\nPerfil de outliers (medias):")
    print(outliers[feature_cols].mean().round(1))
    
    print("\nComparación con medias globales:")
    print(df[feature_cols].mean().round(1))

---
## 8. Comparación de Resultados: K-Means vs DBSCAN

Resumimos las diferencias clave entre ambos enfoques.

In [ ]:
print("=== Comparación K-Means vs DBSCAN ===")
print(f"K-Means:")
print(f"- Número de clusters: {best_k}")
print(f"- Todos los puntos asignados a un cluster")
print(f"- Forma de clusters: esféricos (por construcción)")

n_clusters_db = len(set(labels_dbscan_final)) - (1 if -1 in labels_dbscan_final else 0)
n_noise_db = list(labels_dbscan_final).count(-1)
print(f"\nDBSCAN:")
print(f"- Número de clusters: {n_clusters_db}")
print(f"- Puntos de ruido: {n_noise_db} ({n_noise_db/len(df):.1%})")
print(f"- Forma de clusters: arbitraria (basada en densidad)")

print("\nVentajas de cada uno:")
print("- K-Means: Simple, rápido, fácil de interpretar. Requiere elegir k.")
print("- DBSCAN: No requiere k, detecta outliers, maneja formas arbitrarias. Requiere ajuste de parámetros.")

---
## 9. Conclusiones

En este notebook hemos aplicado técnicas de aprendizaje no supervisado a un problema real de segmentación de clientes:

✔️ **K-Means**:
- Determinamos el número óptimo de clusters usando el método del codo y el coeficiente de silueta.
- Visualizamos los segmentos en 2D usando PCA.
- Interpretamos los perfiles de cada cluster analizando las medias de las variables.

✔️ **DBSCAN**:
- Exploramos diferentes valores de eps y observamos su efecto.
- Detectamos outliers (puntos de ruido) que podrían representar clientes atípicos.
- Comparamos los resultados con K-Means.

**Lección clave**: La elección entre K-Means y DBSCAN depende de la estructura de los datos y del objetivo del negocio. K-Means es adecuado cuando se esperan clusters de tamaño y forma similar, mientras que DBSCAN es más flexible y puede identificar outliers.

---
**Fin del Notebook de Ejercicios - Semana 9**